# Demo: Classification dans la eCOICOPV2

Ce notebook est un exemple d'utilisation du package[`torchTextClassifiers`](https://github.com/InseeFrLab/torchTextClassifiers) pour la classification de produit (venant de tickets de caisses ici) dans la nomenclature COICOP.
Il permet de montrer la flexibilité du framework pour créer un modèle sur mesure.

On va créer plusieurs modèles différents sur un extrait de données de caisse. Chaque modèle sera évaluée sur un fichier annoté créé lors du test de l'enquête BDF en 2025.

```mermaid
flowchart LR
    A["📄 Reçu brut (presque)"] --> B["🔤 Tokenizer"]
    B --> C["📐 TextEmbedder"]
    C --> D["🧠 ClassificationHead"]
    D --> E["🏷️ code COICOP/score de confiance"]

    style A fill:#e8f4f8,stroke:#2196F3
    style B fill:#fff3e0,stroke:#FF9800
    style C fill:#f3e5f5,stroke:#9C27B0
    style D fill:#e8f5e9,stroke:#4CAF50
    style E fill:#fce4ec,stroke:#E91E63
```


## 0. Setup & Installation

In [ ]:
# Install the package (uncomment if needed)
# !pip install torchTextClassifiers
# For CamemBERT tokenizer support:
# !pip install torchTextClassifiers[huggingface]
# Or install all extras:
%pip install torchTextClassifiers[huggingface]
%pip install torchTextClassifiers[explainability]


# DuckDB is needed to read the encrypted parquet from S3
%pip install duckdb scikit-learn captum 

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import os
import numpy as np
import pandas as pd
import duckdb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score

# torchTextClassifiers imports
from torchTextClassifiers import torchTextClassifiers, ModelConfig, TrainingConfig
from torchTextClassifiers.tokenizers import NGramTokenizer

## 1.Données utilisées

Les données d'entrainement sont un extrait des données de caisse, la description d'un code barre (ean) associé à un code coicop.



In [ ]:
#La clé pour les données DDC trouvable ici https://datalab.sspcloud.fr/my-secrets/projet-budget-famille?openFile=s3

%env DDC_KEY=secret

In [ ]:
# --- Load the encrypted parquet from S3 with DuckDB ---
S3_PATH = "s3://projet-budget-famille/data/ddc_raw/training_dataset_ddc.parquet"
FOOTER_KEY_NAME = "my_key"       # name used to register the key in DuckDB
FOOTER_KEY_VALUE = os.environ['DDC_KEY']  # Replace with the actual encryption key

con = duckdb.connect()

# Install and load the httpfs extension for S3 access
con.execute("INSTALL httpfs; LOAD httpfs;")

# Configure S3 credentials

con.execute(
 f"""CREATE SECRET secret_ls3 (
            TYPE S3,
            KEY_ID '{os.environ["AWS_ACCESS_KEY_ID"]}',
            SECRET '{os.environ["AWS_SECRET_ACCESS_KEY"]}',
            ENDPOINT '{os.environ["AWS_S3_ENDPOINT"]}',
            SESSION_TOKEN '{os.environ["AWS_SESSION_TOKEN"]}',
            REGION 'us-east-1',
            URL_STYLE 'path',
            SCOPE 's3://travail/projet-ml-classification-bdf'
        );"""
)

# Register the parquet encryption key
con.execute(f"PRAGMA add_parquet_key('{FOOTER_KEY_NAME}', '{FOOTER_KEY_VALUE}');")

# Read the encrypted parquet
df = con.execute(
    f"SELECT product_orig, product, code8 as code FROM read_parquet('{S3_PATH}', "
    f"encryption_config = {{footer_key: '{FOOTER_KEY_NAME}'}})"
).fetchdf()

con.close()




df, _ = train_test_split(df, test_size=0.5, random_state=42, stratify=df["code"])

print(f"Dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")

In [ ]:
df.sample(10)

Les données de validations utilisées viennent d'un pilote de l'enquête BDF en 2025 

In [ ]:
validation_df = pd.read_parquet('s3://projet-budget-famille/data/output-annotation-consolidated/annotations-consolidated-2026-03-09/raw_preproc.parquet')
validation_df = validation_df.drop(validation_df[validation_df['code'].str.len()<8].index)
validation_df = validation_df.drop(validation_df[validation_df['bdf_2017']].index)
validation_df = validation_df.drop(validation_df[validation_df['code'].str[:2]=='99'].index)
validation_df = validation_df.drop(validation_df[validation_df['code'].str[:2]=='98'].index)

validation_df['code'] = validation_df['code'].str[:8]
validation_df[['product','code','coicop']].sample(10)

In [ ]:
coicop_df = pd.read_csv('s3://projet-budget-famille/data/input-annotation/20260130-coicop_et_codes_techniques.csv', sep=';');
coicop_df.head()

In [ ]:
# ─── Derive COICOP levels 1–3 by truncating the level-4 code ───
# COICOP v2 codes typically look like "01.1.1.1" or "01.1.1.1.1"
# We split on "." and take progressively more segments

parts = df["code"].str.split(".")
df["code_level1"] = parts.str[0]
df["code_level2"] = parts.str[:2].str.join(".")
df["code_level3"] = parts.str[:3].str.join(".")
df["code_level4"] = df["code"]   # the original column is already level 4

print("COICOP hierarchy derived from level 4:")
for lvl in [1, 2, 3, 4]:
    col = f"code_level{lvl}"
    n = df[col].nunique()
    examples = sorted(df[col].unique())[:6]
    print(f"  Level {lvl} — {n:>3} classes  (e.g. {examples}{'...' if n > 6 else ''})")

### Data split

We split the data into **train** and **validation** sets. The same split is used throughout all experiments for fair comparison.


In [ ]:
# ─── Encode labels as integers (required by torchTextClassifiers) ───
label_encoder_l4 = LabelEncoder()
df["label_l4"] = label_encoder_l4.fit_transform(df["code_level4"])

# Train/val split
df_train, df_val = train_test_split(df, test_size=0.2, random_state=42, stratify=df["label_l4"])
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)

print(f"Train: {len(df_train)} | Val: {len(df_val)}")
print(f"Number of classes (level 4): {df['label_l4'].nunique()}")

---

## Part 1: Modèle Basique à la FastText

On commence par l'approche basique: un modèle de type fasttext :

```mermaid
flowchart TD
    subgraph "NGramTokenizer"
        A["'riz basmati'"] --> B["character n-grams<br/>'&lt;ri', 'riz', 'iz&gt;', ..."]
        B --> C["hash → token IDs"]
    end
    C --> D["TextEmbedder"]
    D --> E["ClassificationHead"]
    E --> F["COICOP level 4"]

    style A fill:#e8f4f8,stroke:#2196F3
    style D fill:#f3e5f5,stroke:#9C27B0
    style E fill:#e8f5e9,stroke:#4CAF50
    style F fill:#fce4ec,stroke:#E91E63
```


In [ ]:
# ─── Step 1: Create and train the NGram tokenizer ───
ngram_tokenizer = NGramTokenizer(
    num_tokens=20_000,    # hash bucket size for character n-grams
    min_count=2,           # minimum word frequency to include in vocabulary
    min_n=3,               # minimum n-gram length
    max_n=5,               # maximum n-gram length
    len_word_ngrams=1,     # include word-level unigrams
    training_text=df_train["product"].tolist(),  # train on the training set
)

print(f"Vocabulary size: {ngram_tokenizer.vocab_size:,}")
print(f"Number of known words: {ngram_tokenizer.nwords:,}")

In [ ]:
ngram_tokenizer.tokenize('auchan cornichons extra fins')

In [ ]:
# ─── Step 2: Configure the model ───
num_classes = df["label_l4"].nunique()

model_config = ModelConfig(
    embedding_dim=10,       # size of the text embedding
    num_classes=num_classes,  # COICOP level 4 classes
)

# ─── Step 3: Create the classifier ───
classifier_ngram = torchTextClassifiers(
    tokenizer=ngram_tokenizer,
    model_config=model_config,
)

print(classifier_ngram)

In [ ]:
# ─── Step 4: Define training configuration ───
training_config = TrainingConfig(
    num_epochs=5,
    batch_size=64,
    lr=1e-3,
    patience_early_stopping=3,
    num_workers=32,  # set to 0 for notebooks / small datasets
    save_path="model_ngram"
)

In [ ]:
# ─── Step 5: Prepare data as numpy arrays ───
# torchTextClassifiers expects:
#   X: numpy array where the first column is text (and optional subsequent columns are categorical features)
#   Y: numpy array of integer labels

X_train = df_train["product"].values
Y_train = df_train["label_l4"].values

X_val = df_val["product"].values
Y_val = df_val["label_l4"].values

print(f"X_train shape: {X_train.shape}, Y_train shape: {Y_train.shape}")
print(f"X_val shape:   {X_val.shape},   Y_val shape:   {Y_val.shape}")

In [ ]:
# ─── Step 6: Train! ───
classifier_ngram.train(
    X_train=np.asarray(X_train),
    y_train=np.asarray(Y_train),
    X_val=np.asarray(X_val),
    y_val=np.asarray(Y_val),
    training_config=training_config,
    verbose=True,
)

In [ ]:
# ─── Step 7: Evaluate ───
results = classifier_ngram.predict(np.asarray(validation_df['product']), top_k=1)
preds = label_encoder_l4.inverse_transform(results["prediction"].numpy().flatten())
confidence = results["confidence"].numpy().flatten()

acc = accuracy_score(np.asarray(validation_df['code']), preds)
print(f"\n✅ NGram Classifier — Validation Accuracy: {acc:.4f}")
print(f"   Average confidence: {confidence.mean():.4f}")



print("\nClassification Report (sample) by level:")
for level in range(1,5):
    preds_tmp = [pred[:level*2] for pred in preds]
    code_tmp = [code[:level*2] for code in np.asarray(validation_df['code'])]
    print(f"\n\t*******  Level {level}  ***************")
    print(classification_report(code_tmp, preds_tmp, zero_division=0))


---

## Part 2: Swapping the Tokenizer 

One of the key strengths of `torchTextClassifiers` is that the tokenizer is fully **pluggable**. We can replace the n-gram tokenizer with any HuggingFace-compatible tokenizer.



In [ ]:
# We use HuggingFaceTokenizer.load_from_pretrained to load a pretrained tokenizer
from torchTextClassifiers.tokenizers import WordPieceTokenizer

# Load the CamemBERT tokenizer
# The output_dim parameter controls the maximum sequence length (context window)
# 1) Create and train the tokenizer on your corpus
wordpiece_tokenizer = WordPieceTokenizer(
    vocab_size=10000,     # target vocabulary size
    output_dim=32,       # max sequence length (receipt texts are short)
)

wordpiece_tokenizer.train(training_corpus=df_train["product"].tolist())

print(f"Vocabulary size: {wordpiece_tokenizer.vocab_size}")

# 2) See what tokenization looks like
output = wordpiece_tokenizer.tokenize("top budget sach batonnets eur")
print("Token IDs:", output.input_ids)
print("Attention mask:", output.attention_mask)

# Decode back to readable tokens
tokens = wordpiece_tokenizer.tokenizer.tokenize("top budget sach batonnets eur")
print("Tokens:", tokens)

In [ ]:
# ─── Build a new classifier with the CamemBERT tokenizer ───
model_config_wordpiece = ModelConfig(
    embedding_dim=10,       # same embedding size for fair comparison
    num_classes=num_classes,
)

classifier_wordpiece = torchTextClassifiers(
    tokenizer=wordpiece_tokenizer,
    model_config=model_config_wordpiece,
)

print(classifier_wordpiece)

In [ ]:
wordpiece_tokenizer.tokenize("auchan lentille")

In [ ]:
# ─── Train with the same configuration ───
training_config_wordpiece = TrainingConfig(
    num_epochs=5,
    batch_size=64,
    lr=1e-3,
    patience_early_stopping=3,
    num_workers=32,
    save_path="model_wordpiece",
    trainer_params={"enable_progress_bar": True},
)

classifier_wordpiece.train(
    X_train=np.asarray(X_train),
    y_train=np.asarray(Y_train),
    X_val=np.asarray(X_val),
    y_val=np.asarray(Y_val),
    training_config=training_config_wordpiece,
    verbose=True,
)

In [ ]:
# ─── Evaluate the CamemBERT-tokenized model ───
results_cam = classifier_wordpiece.predict(np.asarray(validation_df['product']), top_k=1)
preds_cam = label_encoder_l4.inverse_transform(results_cam["prediction"].numpy().flatten())

acc_cam = accuracy_score(np.asarray(validation_df['code']), preds_cam)
print(f"\n✅ Wordpiece Tokenizer — Validation Accuracy: {acc_cam:.4f}")

# ─── Side-by-side comparison ───
print("\n📊 Tokenizer comparison:")
print(f"  NGram tokenizer:     {acc:.4f}")
print(f"  WordPiece tokenizer: {acc_cam:.4f}")

---

## Part 3: "Fine-Tuning" 

Utiliser de nouvelles données pour affiner l'entrainement.



```mermaid
flowchart TB
    subgraph "Phase 1: Initial Training"
        D1["📦 Original dataset"] --> M1["🧠 Trained model<br/>weights = W₁"]
    end

    subgraph "Phase 2: Fine-Tuning"
        D2["📦 New data<br/>(different naming)"] --> M2["🧠 Fine-tuned model<br/>weights = W₂"]
        M1 -.->|"start from W₁"| M2
    end

    style D1 fill:#e3f2fd,stroke:#2196F3
    style D2 fill:#fff3e0,stroke:#FF9800
    style M1 fill:#e8f5e9,stroke:#4CAF50
    style M2 fill:#fce4ec,stroke:#E91E63
```


In [ ]:
# ─── Simulate "new" data for fine-tuning ───
# In practice, this would be a fresh batch of labelled receipts.
# Here we simulate it by sampling a small held-out portion of val data
# that represents "new" data arriving after initial deployment.

# Take a random 20% of the validation set as "new incoming data"
#validation_df = validation_df.drop(validation_df[validation_df['code'].str[:2]=='98'].index)
df_new =validation_df.sample(frac=0.2, random_state=123).reset_index(drop=True)
X_new = df_new["product"].values
Y_new = label_encoder_l4.transform(df_new["code"].values)

print(f"New data: {len(df_new)} samples across {df_new['code'].nunique()} COICOP codes")
print("(In practice, replace this with genuinely new labelled data)")

In [ ]:
# ─── Evaluate the model on new data BEFORE fine-tuning ───
results_before = classifier_ngram.predict(X_new, top_k=1)
preds_before = label_encoder_l4.inverse_transform(results_before["prediction"].numpy().flatten())
acc_before = accuracy_score(label_encoder_l4.inverse_transform(Y_new), preds_before)
print(f"Before fine-tuning — Accuracy on new data: {acc_before:.4f}")

In [ ]:
# ─── Fine-tune the model on the new data ───
# We use a lower learning rate and fewer epochs for fine-tuning
finetuning_config = TrainingConfig(
    num_epochs=5,
    batch_size=32,
    lr=1e-4,               # lower LR for fine-tuning (10x less than initial)
    patience_early_stopping=3,
    num_workers=0,
    save_path="model_ngram_finetuned",
    trainer_params={"enable_progress_bar": True},
)

# Fine-tune: simply call .train() again on the same classifier instance
# The model starts from the current (trained) weights
classifier_ngram.train(
    X_train=np.asarray(X_new),
    y_train=np.asarray(Y_new),
    training_config=finetuning_config,
    verbose=True,
)

In [ ]:
# ─── Evaluate after fine-tuning ───
results_after = classifier_ngram.predict(X_new, top_k=1)
preds_after = results_after["prediction"].numpy().flatten()
acc_after = accuracy_score(Y_new, preds_after)

print(f"\n📊 Fine-tuning impact on new data:")
print(f"  Before: {acc_before:.4f}")
print(f"  After:  {acc_after:.4f}")

# Also check we haven't catastrophically forgotten the original data
results_orig = classifier_ngram.predict(X_val, top_k=1)
preds_orig = results_orig["prediction"].numpy().flatten()
acc_orig = accuracy_score(Y_val, preds_orig)
print(f"\n  Original val set accuracy (post-finetuning): {acc_orig:.4f}")

---

## Part 4: Hierarchical Classification

The COICOP nomenclature is **hierarchical**: level 1 → level 2 → level 3 → level 4. A code like `01.1.2.1` decomposes as:

| Level | Code | Meaning |
|-------|------|---------|
| 1 | `01` | Food and non-alcoholic beverages |
| 2 | `01.1` | Food |
| 3 | `01.1.2` | Meat |
| 4 | `01.1.2.1` | Beef |

A **hierarchical approach** trains a separate classifier at each level, and feeds the prediction from the preceding level as an additional **categorical feature** to the next level's classifier.

```mermaid
flowchart TB
    TEXT["📄 Receipt text"]
    
    TEXT --> C1["🧠 Classifier Level 1"]
    C1 -->|"pred: 01"| P1["Level 1 prediction"]
    
    TEXT --> C2["🧠 Classifier Level 2"]
    P1 -->|"as categorical feature"| C2
    C2 -->|"pred: 01.1"| P2["Level 2 prediction"]
    
    TEXT --> C3["🧠 Classifier Level 3"]
    P2 -->|"as categorical feature"| C3
    C3 -->|"pred: 01.1.2"| P3["Level 3 prediction"]
    
    TEXT --> C4["🧠 Classifier Level 4"]
    P3 -->|"as categorical feature"| C4
    C4 -->|"pred: 01.1.2.1"| P4["🏷️ Final COICOP code"]

    style TEXT fill:#e8f4f8,stroke:#2196F3
    style C1 fill:#e8f5e9,stroke:#4CAF50
    style C2 fill:#e8f5e9,stroke:#4CAF50
    style C3 fill:#e8f5e9,stroke:#4CAF50
    style C4 fill:#e8f5e9,stroke:#4CAF50
    style P4 fill:#fce4ec,stroke:#E91E63
```

**Advantages:**
- Each classifier focuses on a simpler task (fewer classes)
- Errors at higher levels constrain the search space for lower levels
- Mirrors the logical structure of the nomenclature


### 4.1 — Prepare label encoders for each level

In [ ]:
# ─── Label encoders for each level ───
label_encoders = {}
for level in [1, 2, 3, 4]:
    col = f"code_level{level}"
    le = LabelEncoder()
    df_train[f"label_l{level}"] = le.fit_transform(df_train[col])
    df_val[f"label_l{level}"] = le.transform(df_val[col])
    label_encoders[level] = le
    print(f"Level {level}: {len(le.classes_)} classes → {list(le.classes_[:5])}{'...' if len(le.classes_) > 5 else ''}")

### 4.2 — Level 1 Classifier (text only)

At the top of the hierarchy, we classify with text alone — no parent-level predictions available.


In [ ]:
# ─── Level 1: text-only classifier ───
num_classes_l1 = df_train["label_l1"].nunique()

tokenizer_l1 = NGramTokenizer(
    num_tokens=100_000, min_count=2, min_n=3, max_n=5,
    len_word_ngrams=1,
    training_text=df_train["product"].tolist(),
)

classifier_l1 = torchTextClassifiers(
    tokenizer=tokenizer_l1,
    model_config=ModelConfig(embedding_dim=10, num_classes=num_classes_l1),
)

training_cfg = TrainingConfig(
    num_epochs=5, batch_size=64, lr=1e-3,
    patience_early_stopping=3, num_workers=0,
    save_path="model_hier_l1",
    trainer_params={"enable_progress_bar": True},
)

classifier_l1.train(
    X_train=np.asarray(df_train["product"].values),
    y_train=np.asarray(df_train["label_l1"].values),
    X_val=np.asarray(df_val["product"].values),
    y_val=np.asarray(df_val["label_l1"].values),
    training_config=training_cfg,
    verbose=True,
)

# Evaluate level 1
res_l1 = classifier_l1.predict(np.asarray(df_val["product"].values), top_k=1)
preds_l1 = res_l1["prediction"].numpy().flatten()
acc_l1 = accuracy_score(df_val["label_l1"].values, preds_l1)
print(f"\n✅ Level 1 Accuracy: {acc_l1:.4f}")

### 4.3 — Levels 2–4: Text + Prediction from Preceding Level

For levels 2, 3, and 4, we pass the **predicted label from the preceding level** as a categorical variable.

The `torchTextClassifiers` package supports mixed inputs (text + categorical) natively:
- `X` must be a numpy array where column 0 is text and columns 1+ are integer-encoded categorical variables
- `ModelConfig` must specify `categorical_vocabulary_sizes` and `categorical_embedding_dims`

```mermaid
flowchart LR
    subgraph "Input for Level k"
        T["Text<br/>(column 0)"]
        P["Pred Level k-1<br/>(column 1, integer)"]
    end

    subgraph "Model"
        TE["TextEmbedder"]
        CVN["CategoricalVariableNet"]
        CH["ClassificationHead"]
    end

    T --> TE
    P --> CVN
    TE --> CH
    CVN --> CH

    style T fill:#e3f2fd,stroke:#2196F3
    style P fill:#fff3e0,stroke:#FF9800
    style CVN fill:#f1f8e9,stroke:#8BC34A
```


In [ ]:
# --- Helper function: train one level with parent prediction as feature ---
def train_hierarchical_level(
    level: int,
    df_train: pd.DataFrame,
    df_val: pd.DataFrame,
    parent_preds_train: np.ndarray,
    parent_preds_val: np.ndarray,
    parent_num_classes: int
):
    """
    Train a classifier for a given COICOP level, using the parent level
    predictions as a categorical variable.

    Args:
        level: COICOP level to classify (2, 3, or 4)
        df_train, df_val: DataFrames with product and label_lN columns
        parent_preds_train, parent_preds_val: parent-level predictions (int)
        parent_num_classes: number of classes at the parent level

    Returns:
        classifier, predictions on val set, accuracy
    """
    num_classes=df_train[f"label_l{level}"].nunique()

    # Create tokenizer for this level
    tokenizer = NGramTokenizer(
        num_tokens=100_000, min_count=2, min_n=3, max_n=5,
        len_word_ngrams=1,
        training_text=df_train["product"].tolist(),
    )

    # Model config with categorical variable (parent prediction)
    model_cfg = ModelConfig(
        embedding_dim=10,
        num_classes=num_classes,
        categorical_vocabulary_sizes=[parent_num_classes],   # one cat variable
        categorical_embedding_dims=[8],                      # its embedding size
    )

    classifier = torchTextClassifiers(
        tokenizer=tokenizer,
        model_config=model_cfg,
    )

    # Prepare X as (text, parent_prediction) -- 2D numpy array
    X_train_hier = np.column_stack([
        df_train["product"].values,
        parent_preds_train.astype(str),  # will be cast to int internally
    ])
    X_val_hier = np.column_stack([
        df_val["product"].values,
        parent_preds_val.astype(str),
    ])

    training_cfg = TrainingConfig(
        num_epochs=5, batch_size=64, lr=1e-3,
        patience_early_stopping=3, num_workers=0,
        save_path=f"model_hier_l{level}",
        trainer_params={"enable_progress_bar": True},
    )

    classifier.train(
        X_train=np.asarray(X_train_hier),
        y_train=np.asarray(df_train[f"label_l{level}"].values),
        X_val=np.asarray(X_val_hier),
        y_val=np.asarray(df_val[f"label_l{level}"].values),
        training_config=training_cfg,
        verbose=True,
    )

    # Predict
    res = classifier.predict(np.asarray(X_val_hier), top_k=1)
    preds = res["prediction"].numpy().flatten()
    acc = accuracy_score(df_val[f"label_l{level}"].values, preds)

    return classifier, preds, acc

In [ ]:
# ─── Level 2: text + level 1 predictions ───
# For training, we use ground-truth level 1 labels (teacher forcing)
# For validation, we use the model's level 1 predictions (realistic cascade)

classifier_l2, preds_l2, acc_l2 = train_hierarchical_level(
    level=2,
    df_train=df_train,
    df_val=df_val,
    parent_preds_train=np.asarray(df_train["label_l1"].values),  # ground truth for training
    parent_preds_val=preds_l1,                        # model predictions for val
    parent_num_classes=num_classes_l1
)
print(f"\n✅ Level 2 Accuracy: {acc_l2:.4f}")

In [ ]:
# ─── Level 3: text + level 2 predictions ───
num_classes_l2 = df_train["label_l2"].nunique()

classifier_l3, preds_l3, acc_l3 = train_hierarchical_level(
    level=3,
    df_train=df_train,
    df_val=df_val,
    parent_preds_train=df_train["label_l2"].values,
    parent_preds_val=preds_l2,
    parent_num_classes=num_classes_l2,
)
print(f"\n✅ Level 3 Accuracy: {acc_l3:.4f}")

In [ ]:
# ─── Level 4: text + level 3 predictions ───
num_classes_l3 = df_train["label_l3"].nunique()

classifier_l4, preds_l4, acc_l4 = train_hierarchical_level(
    level=4,
    df_train=df_train,
    df_val=df_val,
    parent_preds_train=df_train["label_l3"].values,
    parent_preds_val=preds_l3,
    parent_num_classes=num_classes_l3,
)
print(f"\n✅ Level 4 Accuracy: {acc_l4:.4f}")

### 4.4 — Summary of Hierarchical Results

In [ ]:
# ─── Summary ───
print("=" * 50)
print("HIERARCHICAL CLASSIFICATION RESULTS")
print("=" * 50)
print(f"{'Level':<10} {'Classes':<10} {'Accuracy':<10}")
print("-" * 30)
for level, acc_val, n_cls in [
    (1, acc_l1, num_classes_l1),
    (2, acc_l2, df_train["label_l2"].nunique()),
    (3, acc_l3, df_train["label_l3"].nunique()),
    (4, acc_l4, df_train["label_l4"].nunique()),
]:
    print(f"Level {level:<5} {n_cls:<10} {acc_val:<10.4f}")

print("-" * 30)
print(f"\nFlat L4 classifier (Part 1):       {acc:.4f}")
print(f"Hierarchical L4 classifier (Part 4): {acc_l4:.4f}")

---

## Saving and Loading Models

Every trained classifier can be saved and loaded with a simple API:


In [ ]:
# ─── Save ───
classifier_ngram.save("my_saved_model")

# ─── Load ───
loaded_model = torchTextClassifiers.load("my_saved_model")

# Verify it works
res_loaded = loaded_model.predict(X_val[:5], top_k=1)
print("Predictions from loaded model:")

for x in X_val[:10]:
    pred = label_encoder_l4.inverse_transform(loaded_model.predict(np.asarray([x]))['prediction'])[0]
    conf = loaded_model.predict(np.asarray([x]))['confidence'].item()
    coicop_lib = coicop_df[coicop_df['Code']==pred]['Libelle'].iloc[0]
    print(f"{x} -> {pred} ({coicop_lib})  | confiance: {conf:0.2%})")




### Explicabilité


In [ ]:
np.concatenate?

In [ ]:
from torchTextClassifiers.utilities.plot_explainability import (
    map_attributions_to_char,
    plot_attributions_at_char,
    plot_attributions_at_word,
    get_id_to_word,
    figshow
)
import torch
from matplotlib import pyplot as plt


def map_attributions_to_word(attributions, text, word_ids, offsets):
    """
    Maps token-level attributions to word-level attributions based on word IDs.
    Args:
        attributions (np.ndarray): Array of shape (top_k, seq_len) or (seq_len,) containing token-level attributions.
               Output from:
                >>> ttc.predict(X, top_k=top_k, explain=True)["attributions"]
        word_ids (list of int or None): List of word IDs for each token in the original text.
                Output from:
                >>> ttc.predict(X, top_k=top_k, explain=True)["word_ids"]

    Returns:
        np.ndarray: Array of shape (top_k, num_words) containing word-level attributions.
            num_words is the number of unique words in the original text.
    """
    
    word_ids = np.array(word_ids)
    words = get_id_to_word(text, word_ids, offsets)

    # Convert None to -1 for easier processing (PAD tokens)
    word_ids_int = np.array([x if x is not None else -1 for x in word_ids], dtype=int)

    # Filter out PAD tokens from attributions and word_ids
    attributions = attributions[
        torch.arange(attributions.shape[0])[:, None],
        torch.tensor(np.where(word_ids_int != -1)[0])[None, :],
    ]
    word_ids_int = word_ids_int[word_ids_int != -1]
    unique_word_ids = np.unique(word_ids_int)
    num_unique_words = len(unique_word_ids)

    top_k = attributions.shape[0]
    attr_with_word_id = np.concatenate(
        (attributions[:, :, None], np.tile(word_ids_int[None, :], reps=(top_k, 1))[:, :, None]),
        axis=-1,
    )  # top_k, seq_len, 2
    # last dim is 2: 0 is the attribution of the token, 1 is the word_id the token is associated to

    word_attributions = np.zeros((top_k, num_unique_words))
    for word_id in unique_word_ids:
        mask = attr_with_word_id[:, :, 1] == word_id  # top_k, seq_len
        word_attributions[:, word_id] = (attr_with_word_id[:, :, 0] * mask).sum(
            axis=1
        )  # zero-out non-matching tokens and sum attributions for all tokens belonging to the same word

    # assert word_attributions.sum(axis=1) == attributions.sum(axis=1), "Sum of word attributions per top_k must equal sum of token attributions per top_k."
    return words, np.exp(word_attributions) / np.sum(
        np.exp(word_attributions), axis=1, keepdims=True
    )  # softmax normalization


def predict_and_explain(text="danette chocolat", top_k=10):
    model=classifier_ngram
    prediction = model.predict(np.asarray([text]),top_k=top_k, explain_with_captum=True)
    for i in range(top_k):
        pred = label_encoder_l4.inverse_transform(prediction['prediction'].reshape(-1))[i]
        conf = prediction['confidence'][0,i].item()
        coicop_lib = coicop_df[coicop_df['Code']==pred]['Libelle'].iloc[0]
        print(f"--> {pred} ({coicop_lib})")
        print(f"\t confiance: {conf:0.2%}")
        if top_k==1:
            offsets = prediction["offset_mapping"][i]  # seq_len, 2
            attributions = prediction["captum_attributions"][i]  # top_k, seq_len
            word_ids = prediction["word_ids"][i]  # seq_len
            words, word_attributions = map_attributions_to_word(attributions, text, word_ids, offsets)
            plot_attributions_at_word(text=text, words=list(words.values()), attributions_per_word=word_attributions)

#predict_and_explain(top_k=1)





In [ ]:
from ipywidgets import interact, interactive, fixed, interact_manual
import ipywidgets as widgets

interact(predict_and_explain, text='danette chocolat', top_k=widgets.IntSlider(min=1, max=10, step=1, value=1))

print()